# 흉부 X-ray 폐렴 분류 최종 노트북

EfficientNet-B0, ResNet50, ConvNeXt-Tiny 3개 모델을 학습해 폐렴 확률을 평균 내고, 임계값 0.3을 적용해 제출 파일을 만듭니다. 최종 리더보드 정확도는 0.96795(604/624장)입니다.

## 0. 환경 준비 (드라이브의 zip을 코랩으로 복사해 압축 해제)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 드라이브의 zip 경로는 본인 위치로 수정
!cp "/content/drive/MyDrive/강의/데이콘 해커톤/open.zip" /content/data.zip
!unzip -q -o /content/data.zip -d /content/data
!ls /content/data


Mounted at /content/drive
sample_submission.csv  test  test.csv  train  train.csv


## 1. 설정

In [ ]:
import os, random, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

DATA_DIR = "/content/data"
IMG_DIR_TRAIN = "/content/data/train"
IMG_DIR_TEST = "/content/data/test"

SEED, IMG_SIZE, BATCH, EPOCHS = 42, 256, 32, 10
# 모델별 학습률 (이름 -> lr)
MODELS = {"efficientnet_b0": 3e-4, "resnet50": 2e-4, "convnext_tiny": 1e-4}
device = "cuda"

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


## 2. 데이터 / 증강

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train_df.label.value_counts())

def load_img(path):
    # 촬영 조건 차이를 줄이기 위해 대비를 정규화한 뒤 3채널로 변환
    img = Image.open(path).convert("L")
    img = ImageOps.autocontrast(img, cutoff=1)
    return img.convert("RGB")

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
    transforms.ToTensor(), norm,
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.1)),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), norm])

class XrayDS(Dataset):
    def __init__(self, df, img_dir, tf, has_label=True):
        self.df, self.img_dir, self.tf, self.has_label = df.reset_index(drop=True), img_dir, tf, has_label
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = self.tf(load_img(os.path.join(self.img_dir, r.file_name)))
        return (x, int(r.label)) if self.has_label else x

def loader(df, d, tf, shuffle=False, has_label=True):
    return DataLoader(XrayDS(df, d, tf, has_label), BATCH, shuffle=shuffle,
                      num_workers=2, pin_memory=True)

te_dl = loader(test_df, IMG_DIR_TEST, eval_tf, has_label=False)

label
1    3875
0    1341
Name: count, dtype: int64


## 3. 모델 / 학습 / 추론 함수

In [ ]:
def build(name):
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m.to(device)

@torch.no_grad()
def predict_proba(model, dl, tta=True):
    model.eval(); out = []
    for x in dl:
        if isinstance(x, (list, tuple)): x = x[0]
        x = x.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            p = model(x).float().softmax(1)
            if tta: p = (p + model(torch.flip(x, dims=[3])).float().softmax(1)) / 2
        out.append(p[:, 1].cpu().numpy())
    return np.concatenate(out)

def train_one(name, lr, seed):
    seed_all(seed)
    tr_df, va_df = train_test_split(train_df, test_size=0.1, stratify=train_df.label, random_state=seed)
    tr_dl = loader(tr_df, IMG_DIR_TRAIN, train_tf, shuffle=True)
    va_dl = loader(va_df, IMG_DIR_TRAIN, eval_tf)
    model = build(name)
    cnt = train_df.label.value_counts().sort_index().values
    w = torch.tensor(cnt.sum() / (2 * cnt), dtype=torch.float).to(device)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EPOCHS * len(tr_dl), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda")
    for ep in range(EPOCHS):
        model.train(); tot = 0
        for x, y in tr_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(x)
        vp = predict_proba(model, va_dl, tta=False)
        acc = accuracy_score(va_df.label.values, (vp > 0.5).astype(int))
        print(f"{name} [{ep+1}/{EPOCHS}] loss {tot/len(tr_dl.dataset):.4f} | val acc {acc:.4f}")
    # val은 train과 분포가 같아 점수가 부풀려지므로 best 선택 없이 마지막 에폭 모델을 사용
    return model

## 4. 앙상블 학습과 테스트 확률 저장

In [ ]:
all_probs = {}
for i, (name, lr) in enumerate(MODELS.items()):
    model = train_one(name, lr, SEED + i)
    p = predict_proba(model, te_dl)
    all_probs[name] = p
    np.save(f"/content/probs_{name}.npy", p)   # 필요하면 드라이브로 복사해 두세요
    print(name, "예측 폐렴 비율:", (p > 0.5).mean().round(3))
    del model; torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 80.2MB/s]


efficientnet_b0 [1/10] loss 0.4819 | val acc 0.9521
efficientnet_b0 [2/10] loss 0.3523 | val acc 0.9234
efficientnet_b0 [3/10] loss 0.3242 | val acc 0.9808
efficientnet_b0 [4/10] loss 0.3096 | val acc 0.9885
efficientnet_b0 [5/10] loss 0.3059 | val acc 0.9866
efficientnet_b0 [6/10] loss 0.2982 | val acc 0.9808
efficientnet_b0 [7/10] loss 0.2879 | val acc 0.9904
efficientnet_b0 [8/10] loss 0.2841 | val acc 0.9885
efficientnet_b0 [9/10] loss 0.2791 | val acc 0.9885
efficientnet_b0 [10/10] loss 0.2802 | val acc 0.9885
efficientnet_b0 예측 폐렴 비율: 0.571
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 164MB/s]
/tmp/ipykernel_4913/1630840482.py:44: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


resnet50 [1/10] loss 0.5045 | val acc 0.9464
resnet50 [2/10] loss 0.3532 | val acc 0.9789
resnet50 [3/10] loss 0.3250 | val acc 0.9693
resnet50 [4/10] loss 0.3097 | val acc 0.9751
resnet50 [5/10] loss 0.3009 | val acc 0.9751
resnet50 [6/10] loss 0.2982 | val acc 0.9847
resnet50 [7/10] loss 0.2861 | val acc 0.9808
resnet50 [8/10] loss 0.2815 | val acc 0.9751
resnet50 [9/10] loss 0.2801 | val acc 0.9847
resnet50 [10/10] loss 0.2758 | val acc 0.9847
resnet50 예측 폐렴 비율: 0.599
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:01<00:00, 62.1MB/s]


convnext_tiny [1/10] loss 0.4468 | val acc 0.8736
convnext_tiny [2/10] loss 0.3457 | val acc 0.9789
convnext_tiny [3/10] loss 0.3051 | val acc 0.9847
convnext_tiny [4/10] loss 0.3019 | val acc 0.9828
convnext_tiny [5/10] loss 0.2954 | val acc 0.9885
convnext_tiny [6/10] loss 0.2899 | val acc 0.9847
convnext_tiny [7/10] loss 0.2803 | val acc 0.9847
convnext_tiny [8/10] loss 0.2786 | val acc 0.9866
convnext_tiny [9/10] loss 0.2731 | val acc 0.9866
convnext_tiny [10/10] loss 0.2703 | val acc 0.9866
convnext_tiny 예측 폐렴 비율: 0.611


## 5. 기준 제출 파일 (임계값 0.5)

3모델 평균 확률에 임계값 0.5를 적용한 파일입니다(리더보드 0.95673).

In [ ]:
avg = np.mean(list(all_probs.values()), axis=0)
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
assert (sub.file_name.values == test_df.file_name.values).all()
sub["label"] = (avg > 0.5).astype(int)
sub.to_csv("submission_v2.csv", index=False)
print(sub.label.value_counts())

label
1    373
0    251
Name: count, dtype: int64


## 6. 임계값 분석과 최종 제출 파일 (임계값 0.3)

저장해 둔 확률(`probs_*.npy`)로 임계값을 바꿔 폐렴 예측 장수를 비교하고, 최종 임계값 0.3으로 제출 파일을 만듭니다. 재학습은 필요 없습니다.

In [ ]:
import numpy as np, pandas as pd, glob
P = np.mean([np.load(f) for f in [f"/content/probs_{n}.npy" for n in MODELS]], axis=0)
sub0 = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
base = (P > 0.5)
for th in [0.6, 0.5, 0.4, 0.3, 0.2]:
    pred = (P > th)
    print(f"th={th}: 폐렴 {pred.sum()}장 | 0.5 대비 바뀐 장수 {(pred != base).sum()}")
# 불확실한 구간에 이미지가 몇 장 있는지
print("확률 0.2~0.8 사이:", ((P > 0.2) & (P < 0.8)).sum(), "장")

TH = 0.3   # 최종 선택 (0.5, 0.4, 0.3을 제출해 비교한 결과 0.3이 최고)
sub0["label"] = (P > TH).astype(int)
sub0.to_csv("submission_v2_th.csv", index=False)

th=0.6: 폐렴 359장 | 0.5 대비 바뀐 장수 14
th=0.5: 폐렴 373장 | 0.5 대비 바뀐 장수 0
th=0.4: 폐렴 386장 | 0.5 대비 바뀐 장수 13
th=0.3: 폐렴 392장 | 0.5 대비 바뀐 장수 19
th=0.2: 폐렴 395장 | 0.5 대비 바뀐 장수 22
확률 0.2~0.8 사이: 109 장


## 제출 이력과 최종 선택

| 단계 | 변경 내용 | 리더보드 | 정답 장수 |
| --- | --- | --- | --- |
| v1 | ResNet18 단일 모델, 임계값 0.5 | 0.93589 | 584/624 |
| v2 | 3모델 앙상블 + 강한 증강, 임계값 0.5 | 0.95673 | 597/624 |
| v2, 임계값 0.4 | 폐렴 386장 예측 | 0.96474 | 602/624 |
| **v2, 임계값 0.3 (최종 제출)** | 폐렴 392장 예측 | **0.96795** | **604/624** |
| v3 | 6모델 앙상블, 임계값 0.3 | 0.96474 | 602/624 |

